# PyTorch Mastery for ML Engineer Interviews

A complete, executable reference covering PyTorch fundamentals through staff-level interview problems.
Every section has runnable code with `assert` checks so correctness is machine-verifiable.

### Company relevance
| Company | What they probe | Typical PyTorch question |
|---------|----------------|-------------------------|
| **Meta** | Custom training loops, distributed training (DDP/FSDP), model debugging | Fix a broken Transformer training loop |
| **Tesla** | Custom layers, DataLoader optimization, real-time inference | Implement conv + pooling from nn.Module |
| **Google** | Autograd mechanics, custom loss functions, numerical stability | Custom autograd Function with gradient check |
| **NVIDIA** | Mixed precision, GPU memory optimization, gradient checkpointing | AMP training loop, memory profiling |

### How to use
1. **Run all cells top-to-bottom** — every cell is self-contained after the setup cell.
2. All correctness checks use `assert`; a silent pass means the answer is correct.
3. Shape annotations are included as comments to build the habit of narrating shapes.

---
## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__}  device={device}')

---
## 2. Tensor Fundamentals — Creation, Dtypes, Device, Memory

### Concept + takeaway

PyTorch tensors are similar to NumPy arrays but add:
- **GPU support**: move tensors to GPU with `.to(device)` or `.cuda()`
- **Autograd tracking**: set `requires_grad=True` to track operations for automatic differentiation
- **Interop with NumPy**: `.numpy()` and `torch.from_numpy()` share memory (CPU only)

**Interview takeaway:** always know where your tensor lives (CPU vs GPU) and whether gradients are being tracked.

In [ ]:
# --- 2a. Creation helpers ---
t_zeros = torch.zeros(2, 3)
t_ones  = torch.ones(2, 3, dtype=torch.float32)
t_eye   = torch.eye(4)
t_range = torch.arange(0, 10, 2)
t_lin   = torch.linspace(0, 1, 5)
t_full  = torch.full((3, 3), fill_value=7.0)
t_rand  = torch.randn(3, 4)        # standard normal
t_empty = torch.empty(2, 2)        # uninitialized

assert t_zeros.shape == (2, 3)
assert t_ones.dtype == torch.float32
assert t_eye.shape == (4, 4)
assert list(t_range) == [0, 2, 4, 6, 8]
assert len(t_lin) == 5
assert torch.all(t_full == 7)
print('All creation helpers verified.')

In [ ]:
# --- 2b. Key attributes ---
x = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.int64)
print(f'shape={x.shape}  ndim={x.ndim}  numel={x.numel()}  dtype={x.dtype}  device={x.device}')

assert x.shape == (2, 3)
assert x.ndim == 2
assert x.numel() == 6
assert x.dtype == torch.int64

### Concept + takeaway (dtype casting and device transfer)

- `.to(dtype)` or `.float()`, `.half()`, `.long()` cast between types.
- `.to(device)` moves tensors between CPU and GPU.
- Mismatched devices/dtypes in operations cause runtime errors.

**Interview takeaway:** always ensure tensors are on the same device and compatible dtypes before operations.

In [ ]:
# --- 2c. Dtype casting ---
f64 = torch.tensor([1.1, 2.2, 3.3], dtype=torch.float64)
f32 = f64.float()            # to float32
f16 = f64.half()             # to float16
i64 = f64.long()             # to int64

assert f32.dtype == torch.float32
assert f16.dtype == torch.float16
assert i64.dtype == torch.int64

# Device transfer (works on any machine)
t_cpu = torch.randn(3, 3)
t_dev = t_cpu.to(device)
assert t_dev.device.type == device.type
print(f'Tensor on {t_dev.device}')

### Concept + takeaway (NumPy interop and memory sharing)

- `torch.from_numpy(arr)` creates a tensor that **shares memory** with the numpy array (CPU only).
- `.numpy()` on a CPU tensor also shares memory.
- Mutations in one are visible in the other.

**Interview takeaway:** mention shared memory explicitly — it's a common source of subtle bugs.

In [ ]:
# --- 2d. NumPy interop ---
np_arr = np.array([1.0, 2.0, 3.0])
t_from_np = torch.from_numpy(np_arr)

np_arr[0] = 99.0
assert t_from_np[0].item() == 99.0, 'Shared memory — numpy mutation visible in tensor'

t_back = t_from_np.numpy()
assert np.shares_memory(np_arr, t_back)
print('NumPy interop verified (shared memory).')

---
## 3. Indexing, Reshaping, Contiguous Memory

### Concept + takeaway

PyTorch indexing works like NumPy. Key differences to know:
- `view()` returns a view (must be contiguous); `reshape()` works regardless but may copy.
- After `transpose()` or `permute()`, the tensor may not be contiguous — call `.contiguous()` before `view()`.
- `flatten()` is a convenience for `view(-1)`.

**Interview takeaway:** understand when `.contiguous()` is needed, and prefer `reshape()` when unsure about layout.

In [ ]:
x = torch.arange(12).reshape(3, 4)
print('x:\n', x)

assert x[1, 2] == 6
assert x[-1, -1] == 11
assert x[:, 2].shape == (3,)

# Boolean mask
mask = x > 5
assert torch.all(x[mask] > 5)

# view vs reshape
v = x.view(4, 3)
assert v.data_ptr() == x.data_ptr()   # same memory

# transpose makes tensor non-contiguous
t = x.t()                              # (4, 3)
assert not t.is_contiguous()

try:
    _ = t.view(-1)                     # fails
    assert False
except RuntimeError:
    pass

# Fix: .contiguous() or .reshape()
flat = t.contiguous().view(-1)
assert flat.shape == (12,)
flat2 = t.reshape(-1)                  # always works
assert flat2.shape == (12,)
print('Indexing & contiguous verified.')

---
## 4. Autograd — Automatic Differentiation

### Concept + takeaway

PyTorch builds a dynamic computation graph on the fly. When you call `.backward()` on a scalar loss, gradients flow back through the graph.

Key rules:
- Only **leaf tensors** with `requires_grad=True` accumulate gradients in `.grad`.
- Gradients **accumulate** by default — call `optimizer.zero_grad()` or `tensor.grad.zero_()` before each backward pass.
- `torch.no_grad()` context disables tracking (used for inference and weight updates).
- `.detach()` creates a tensor that shares data but is disconnected from the graph.

**Interview takeaway:** explain the difference between leaf vs non-leaf tensors, and always mention gradient accumulation as a common bug source.

In [ ]:
# Basic autograd
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2 + 3 * x + 1         # y = x^2 + 3x + 1
loss = y.sum()
loss.backward()

# dy/dx = 2x + 3
expected_grad = 2 * torch.tensor([2.0, 3.0]) + 3
assert torch.allclose(x.grad, expected_grad)
print(f'x.grad = {x.grad}  expected = {expected_grad}')

In [ ]:
# Gradient accumulation demo
x = torch.tensor([1.0], requires_grad=True)
for i in range(3):
    loss = (x * 2).sum()
    loss.backward()

assert x.grad.item() == 6.0, 'Gradients accumulated: 2+2+2=6'

# Fix: zero gradients
x.grad.zero_()
loss = (x * 2).sum()
loss.backward()
assert x.grad.item() == 2.0
print('Gradient accumulation demo verified.')

In [ ]:
# torch.no_grad() and .detach()
x = torch.tensor([5.0], requires_grad=True)

with torch.no_grad():
    y = x * 2
assert not y.requires_grad

z = x.detach()
assert z.data_ptr() == x.data_ptr()   # same memory
assert not z.requires_grad
print('no_grad / detach verified.')

---
## 5. nn.Module — Building Models

### Concept + takeaway

All PyTorch models subclass `nn.Module`. Key things to know:
- Define layers in `__init__`, computation in `forward()`.
- `model.parameters()` yields all learnable parameters (recursively).
- `model.train()` and `model.eval()` switch behavior for dropout/batchnorm.
- `model.to(device)` moves all parameters and buffers.

**Interview takeaway:** be fluent in writing a model from scratch, counting parameters, and switching between train/eval modes.

In [ ]:
class SimpleMLP(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, out_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

model = SimpleMLP(10, 32, 5)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# fc1: 10*32 + 32 = 352, fc2: 32*5 + 5 = 165 -> total 517
assert total_params == 517
assert trainable_params == 517

# Forward pass
x = torch.randn(8, 10)     # batch=8, features=10
out = model(x)
assert out.shape == (8, 5)

# Train/eval mode
model.train()
assert model.training
model.eval()
assert not model.training

print(f'Parameters: {total_params}  Output shape: {out.shape}')

### Concept + takeaway (nn.Parameter vs register_buffer)

- `nn.Parameter`: wraps a tensor so `model.parameters()` yields it — these get gradients and are updated by the optimizer.
- `register_buffer`: stores state that should move with the model (e.g. running mean in batchnorm) but is NOT a learnable parameter.

**Interview takeaway:** mention this distinction when asked about custom layers or stateful modules.

In [ ]:
class CustomLayer(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(dim))       # learnable
        self.register_buffer('running_mean', torch.zeros(dim))  # not learnable

    def forward(self, x):
        if self.training:
            self.running_mean = 0.9 * self.running_mean + 0.1 * x.mean(dim=0).detach()
        return x * self.weight

layer = CustomLayer(4)
params = list(layer.parameters())
buffers = list(layer.buffers())

assert len(params) == 1 and params[0].shape == (4,)
assert len(buffers) == 1 and buffers[0].shape == (4,)
assert params[0].requires_grad
assert not buffers[0].requires_grad
print('nn.Parameter vs register_buffer verified.')

---
## 6. Loss Functions and Optimizers

### Concept + takeaway

- `nn.CrossEntropyLoss` expects raw logits (it applies log-softmax internally).
- `nn.MSELoss` for regression.
- Optimizers (SGD, Adam) update parameters via `optimizer.step()`.
- The canonical training pattern: `zero_grad → forward → loss → backward → step`.

**Interview takeaway:** know that CrossEntropyLoss = LogSoftmax + NLLLoss, and always mention the 5-step training loop.

In [ ]:
# CrossEntropyLoss accepts raw logits, not softmax output
logits = torch.randn(4, 10)                     # batch=4, classes=10
labels = torch.tensor([3, 0, 9, 1])             # integer labels

ce_loss = nn.CrossEntropyLoss()
loss = ce_loss(logits, labels)
assert loss.ndim == 0   # scalar

# Manual equivalent: log_softmax + nll_loss
log_probs = F.log_softmax(logits, dim=1)
manual_loss = F.nll_loss(log_probs, labels)
assert torch.allclose(loss, manual_loss)

print(f'CE loss: {loss.item():.4f} (matches manual: {manual_loss.item():.4f})')

In [ ]:
# Full training step demo
model = SimpleMLP(10, 32, 5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

x = torch.randn(16, 10)
y = torch.randint(0, 5, (16,))

# 5-step training pattern
model.train()
initial_loss = None
for step in range(50):
    optimizer.zero_grad()              # 1. zero gradients
    out = model(x)                     # 2. forward
    loss = criterion(out, y)           # 3. compute loss
    if step == 0:
        initial_loss = loss.item()
    loss.backward()                    # 4. backward
    optimizer.step()                   # 5. update weights

final_loss = loss.item()
assert final_loss < initial_loss, 'Loss should decrease with training'
print(f'Loss: {initial_loss:.4f} -> {final_loss:.4f}')

---
## 7. Custom Dataset and DataLoader

### Concept + takeaway

Custom datasets subclass `torch.utils.data.Dataset` and implement:
- `__len__`: total number of samples
- `__getitem__`: return one sample by index

`DataLoader` wraps a dataset and provides batching, shuffling, and multi-process loading.

**Interview takeaway:** be ready to write a custom Dataset from scratch and explain `num_workers`, `pin_memory`, and `collate_fn`.

In [ ]:
class SyntheticDataset(Dataset):
    def __init__(self, n_samples, n_features, n_classes):
        self.X = torch.randn(n_samples, n_features)
        self.y = torch.randint(0, n_classes, (n_samples,))

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = SyntheticDataset(100, 10, 5)
assert len(dataset) == 100

sample_x, sample_y = dataset[0]
assert sample_x.shape == (10,)
assert sample_y.ndim == 0

loader = DataLoader(dataset, batch_size=16, shuffle=True)
batch_x, batch_y = next(iter(loader))
assert batch_x.shape == (16, 10)
assert batch_y.shape == (16,)
print(f'Dataset size: {len(dataset)}  Batch shape: {batch_x.shape}')

---
## 8. Saving and Loading Models

### Concept + takeaway

Two approaches:
- **Save/load state_dict** (recommended): saves only learnable parameters and buffers.
- **Save/load entire model**: uses pickle — fragile across code changes.

**Interview takeaway:** always prefer `state_dict` approach; mention that you also need to save optimizer state for resuming training.

In [ ]:
import tempfile, os

model = SimpleMLP(10, 32, 5)
x_test = torch.randn(4, 10)

model.eval()
with torch.no_grad():
    original_out = model(x_test)

# Save state_dict
with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
    path = f.name
    torch.save(model.state_dict(), path)

# Load into fresh model
model2 = SimpleMLP(10, 32, 5)
model2.load_state_dict(torch.load(path, weights_only=True))
model2.eval()

with torch.no_grad():
    loaded_out = model2(x_test)

assert torch.allclose(original_out, loaded_out)
os.unlink(path)
print('Save/load state_dict verified.')

---
## 9. Frequently Asked Interview Questions — With Code Proofs

### Q1. What is a computation graph and how does autograd work?

PyTorch builds a **directed acyclic graph (DAG)** on-the-fly during forward pass. Each node is an operation, edges are tensors. `.backward()` traverses this graph in reverse, applying the chain rule to compute gradients.

- The graph is **dynamic** — rebuilt each forward pass (unlike TensorFlow 1.x static graphs).
- Only leaf tensors with `requires_grad=True` store `.grad`.
- Non-leaf tensor gradients are freed after backward unless `retain_grad()` is called.

In [ ]:
a = torch.tensor([2.0], requires_grad=True)   # leaf
b = a * 3                                       # non-leaf
c = b + 1
c.backward()

assert a.is_leaf
assert not b.is_leaf
assert a.grad is not None
assert a.grad.item() == 3.0   # dc/da = 3
assert b.grad is None          # non-leaf grad not retained
print('Computation graph / autograd verified.')

### Q2. view() vs reshape() vs contiguous()

- `view()` requires contiguous memory — fast, returns a view.
- `reshape()` works on any tensor — returns view if possible, copies if needed.
- `contiguous()` rearranges memory to be contiguous if it isn't.

**Interview takeaway:** after `transpose`/`permute`, tensor is usually non-contiguous.

In [ ]:
x = torch.arange(12).reshape(3, 4)
assert x.is_contiguous()

xt = x.t()              # transpose -> non-contiguous
assert not xt.is_contiguous()

xt_c = xt.contiguous()  # now contiguous
assert xt_c.is_contiguous()

# reshape always works
flat = xt.reshape(-1)
assert flat.shape == (12,)
print('view / reshape / contiguous verified.')

### Q3. What is the difference between model.eval() and torch.no_grad()?

- `model.eval()` switches modules like Dropout and BatchNorm to inference mode (no dropout, use running stats).
- `torch.no_grad()` disables gradient tracking — saves memory and compute.
- For inference, you need **both**.

In [ ]:
class ModelWithDropout(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 10)
        self.drop = nn.Dropout(0.5)

    def forward(self, x):
        return self.drop(self.fc(x))

m = ModelWithDropout()
x = torch.randn(100, 10)

m.train()
out_train = m(x)
zero_count_train = (out_train == 0).sum().item()

m.eval()
out_eval = m(x)
zero_count_eval = (out_eval == 0).sum().item()

assert zero_count_train > 0, 'Dropout active in train mode'
assert zero_count_eval == 0, 'Dropout disabled in eval mode'

# no_grad for memory savings
m.eval()
with torch.no_grad():
    out_infer = m(x)
    assert not out_infer.requires_grad

print('eval() vs no_grad() verified.')

### Q4. How do you freeze layers for transfer learning?

Set `requires_grad = False` on parameters you don't want to update. The optimizer will skip them.

In [ ]:
model = SimpleMLP(10, 32, 5)

# Freeze first layer
for param in model.fc1.parameters():
    param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen = sum(p.numel() for p in model.parameters() if not p.requires_grad)

# fc2: 32*5 + 5 = 165 trainable; fc1: 10*32 + 32 = 352 frozen
assert trainable == 165
assert frozen == 352

# Only pass trainable params to optimizer
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
print(f'Trainable: {trainable}  Frozen: {frozen}')

### Q5. What is gradient clipping and why is it used?

Gradient clipping prevents exploding gradients by capping gradient norms. Essential for RNNs and Transformers.

In [ ]:
model = SimpleMLP(10, 32, 5)
x = torch.randn(4, 10)
y = torch.randint(0, 5, (4,))

loss = nn.CrossEntropyLoss()(model(x), y)
loss.backward()

# Clip gradients to max norm of 1.0
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print(f'Gradient norm before clip: {total_norm:.4f}')

# Verify all param gradients are bounded
post_clip_norm = torch.sqrt(sum(p.grad.norm()**2 for p in model.parameters() if p.grad is not None))
assert post_clip_norm <= 1.0 + 1e-6
print(f'Gradient norm after clip: {post_clip_norm:.4f}')

---
## 10. Practice Problems — Full Implementations

Each problem is the kind asked in live ML coding rounds at Tesla, Google, NVIDIA, Meta.
Every solution includes `assert` checks.

### P1. Custom Autograd Function (Google/NVIDIA favorite)

Implement a custom differentiable function with forward and backward.

**How to approach:**
- Subclass `torch.autograd.Function`
- Implement static `forward()` and `backward()` methods
- Save tensors needed for backward via `ctx.save_for_backward()`
- Validate with `torch.autograd.gradcheck()`

In [ ]:
class MySigmoid(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x):
        out = 1.0 / (1.0 + torch.exp(-x))
        ctx.save_for_backward(out)
        return out

    @staticmethod
    def backward(ctx, grad_output):
        out, = ctx.saved_tensors
        return grad_output * out * (1 - out)

x = torch.randn(5, requires_grad=True, dtype=torch.float64)
my_sigmoid = MySigmoid.apply

# Compare with built-in
y_custom = my_sigmoid(x)
y_builtin = torch.sigmoid(x)
assert torch.allclose(y_custom, y_builtin)

# Gradient check
assert torch.autograd.gradcheck(my_sigmoid, (x,), eps=1e-6)
print('Custom autograd Function verified with gradcheck.')

### P2. Custom nn.Module — Residual Block

Implement a residual block (skip connection) used in ResNets.

**How to approach:**
- Two conv layers with batch norm and ReLU
- Add input to output (skip connection)
- Handle dimension mismatch with 1x1 conv shortcut

In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(channels)

    def forward(self, x):
        identity = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity
        return F.relu(out)

block = ResidualBlock(16)
x = torch.randn(2, 16, 8, 8)
out = block(x)

assert out.shape == x.shape, 'Residual block preserves shape'

# Verify gradients flow
loss = out.sum()
loss.backward()
assert all(p.grad is not None for p in block.parameters())
print(f'ResidualBlock: {x.shape} -> {out.shape}')

### P3. Training Loop with Learning Rate Scheduler

**How to approach:**
- Use `torch.optim.lr_scheduler` (StepLR, CosineAnnealingLR, etc.)
- Call `scheduler.step()` after each epoch
- Track learning rate changes

In [ ]:
model = SimpleMLP(10, 32, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

lrs = []
for epoch in range(30):
    lrs.append(optimizer.param_groups[0]['lr'])
    scheduler.step()

assert abs(lrs[0] - 0.1) < 1e-6
assert abs(lrs[10] - 0.05) < 1e-6    # halved after 10 steps
assert abs(lrs[20] - 0.025) < 1e-6   # halved again
print(f'LR schedule: {lrs[0]} -> {lrs[10]} -> {lrs[20]}')

### P4. Implement Multi-Head Attention in PyTorch

**How to approach:**
- Project Q/K/V with linear layers
- Split into heads, compute scaled dot-product attention
- Concatenate heads and project output

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_k = d_model // num_heads
        self.num_heads = num_heads
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, x, mask=None):
        B, S, D = x.shape
        Q = self.W_q(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.num_heads, self.d_k).transpose(1, 2)

        scores = (Q @ K.transpose(-2, -1)) / (self.d_k ** 0.5)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        context = (attn @ V).transpose(1, 2).contiguous().view(B, S, D)
        return self.W_o(context)

mha = MultiHeadAttention(d_model=64, num_heads=4)
x = torch.randn(2, 16, 64)
out = mha(x)
assert out.shape == (2, 16, 64)

# Causal mask
causal_mask = torch.tril(torch.ones(16, 16)).unsqueeze(0).unsqueeze(0)
out_causal = mha(x, mask=causal_mask)
assert out_causal.shape == (2, 16, 64)
print(f'MHA output: {out.shape}  (with causal mask: {out_causal.shape})')

### P5. Implement a Transformer Block

**How to approach:**
- Multi-head attention + residual + layer norm
- Feed-forward network + residual + layer norm
- Pre-norm or post-norm variant

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        x = x + self.drop(self.attn(self.norm1(x), mask))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x

block = TransformerBlock(d_model=64, num_heads=4, d_ff=256)
x = torch.randn(2, 16, 64)
out = block(x)
assert out.shape == (2, 16, 64)

loss = out.sum()
loss.backward()
assert all(p.grad is not None for p in block.parameters())
print(f'TransformerBlock: {x.shape} -> {out.shape}')

### P6. Custom Loss Function — Focal Loss

Used in object detection (RetinaNet). Downweights easy examples.

**How to approach:**
- Compute cross-entropy per sample
- Weight by `(1 - p_t)^gamma` where `p_t` is the probability of the true class

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, reduction='mean'):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        p_t = torch.exp(-ce)       # probability of true class
        focal_weight = (1 - p_t) ** self.gamma
        loss = focal_weight * ce
        if self.reduction == 'mean':
            return loss.mean()
        return loss

logits = torch.randn(16, 10, requires_grad=True)
targets = torch.randint(0, 10, (16,))

focal = FocalLoss(gamma=2.0)
loss = focal(logits, targets)
assert loss.ndim == 0
assert loss.item() > 0

loss.backward()
assert logits.grad is not None

# gamma=0 should equal standard CE
focal0 = FocalLoss(gamma=0.0)
ce_std = nn.CrossEntropyLoss()(logits.detach().requires_grad_(False), targets)
focal0_loss = focal0(logits.detach().requires_grad_(False), targets)
assert torch.allclose(focal0_loss, ce_std, atol=1e-5)
print(f'Focal loss: {loss.item():.4f}  (gamma=0 matches CE: {focal0_loss.item():.4f})')

### P7. Weight Initialization

**How to approach:**
- Use `nn.init` functions inside a custom `init_weights` function
- Apply with `model.apply(init_weights)`

In [ ]:
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.Conv2d):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')

model = SimpleMLP(10, 32, 5)
model.apply(init_weights)

# Verify biases are zero
assert torch.all(model.fc1.bias == 0)
assert torch.all(model.fc2.bias == 0)

# Verify weight variance is reasonable for Kaiming
fan_in = model.fc1.weight.shape[1]
expected_std = (2.0 / fan_in) ** 0.5
actual_std = model.fc1.weight.std().item()
assert 0.3 * expected_std < actual_std < 3.0 * expected_std
print(f'Kaiming init: expected_std={expected_std:.4f}  actual_std={actual_std:.4f}')

### P8. Mixed Precision Training (NVIDIA interview topic)

**How to approach:**
- Use `torch.amp.autocast` for forward pass in float16
- Use `torch.amp.GradScaler` to prevent underflow in gradients
- Pattern: autocast forward → scaled backward → scaler.step → scaler.update

In [ ]:
model = SimpleMLP(10, 32, 5).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scaler = torch.amp.GradScaler(device=device.type)

x = torch.randn(16, 10, device=device)
y = torch.randint(0, 5, (16,), device=device)

model.train()
for step in range(5):
    optimizer.zero_grad()
    with torch.amp.autocast(device_type=device.type):
        out = model(x)
        loss = F.cross_entropy(out, y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

print(f'Mixed precision training completed. Final loss: {loss.item():.4f}')

### P9. Gradient Checkpointing (memory optimization)

Trades compute for memory: recompute activations during backward instead of storing them.

**How to approach:**
- Use `torch.utils.checkpoint.checkpoint()` on memory-heavy segments
- Reduces activation memory by ~50-80%, costs ~20-30% extra compute

In [ ]:
from torch.utils.checkpoint import checkpoint

class CheckpointedModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.block1 = nn.Sequential(nn.Linear(10, 64), nn.ReLU())
        self.block2 = nn.Sequential(nn.Linear(64, 64), nn.ReLU())
        self.head = nn.Linear(64, 5)

    def forward(self, x, use_checkpoint=False):
        if use_checkpoint:
            x = checkpoint(self.block1, x, use_reentrant=False)
            x = checkpoint(self.block2, x, use_reentrant=False)
        else:
            x = self.block1(x)
            x = self.block2(x)
        return self.head(x)

model = CheckpointedModel()
x = torch.randn(8, 10)

# Both should produce same output
out_normal = model(x, use_checkpoint=False)
out_ckpt = model(x, use_checkpoint=True)
assert torch.allclose(out_normal, out_ckpt)

# Both should have gradients
loss = out_ckpt.sum()
loss.backward()
assert all(p.grad is not None for p in model.parameters())
print('Gradient checkpointing verified.')

### P10. Forward and Backward Hooks

Hooks let you inspect or modify inputs/outputs/gradients of modules without changing the model code.

**How to approach:**
- `register_forward_hook`: inspect activations
- `register_backward_hook` / `register_full_backward_hook`: inspect gradients

In [ ]:
activations = {}

def save_activation(name):
    def hook(module, input, output):
        activations[name] = output.detach()
    return hook

model = SimpleMLP(10, 32, 5)
handle1 = model.fc1.register_forward_hook(save_activation('fc1'))
handle2 = model.fc2.register_forward_hook(save_activation('fc2'))

x = torch.randn(4, 10)
_ = model(x)

assert 'fc1' in activations and activations['fc1'].shape == (4, 32)
assert 'fc2' in activations and activations['fc2'].shape == (4, 5)

# Clean up hooks
handle1.remove()
handle2.remove()
print(f'Captured activations: fc1={activations["fc1"].shape}  fc2={activations["fc2"].shape}')

### P11. Implement Label Smoothing

Softens target distribution to reduce overconfidence.

**How to approach:**
- Replace one-hot targets with `(1 - alpha) * one_hot + alpha / num_classes`
- Compute cross-entropy against soft targets

In [ ]:
class LabelSmoothingLoss(nn.Module):
    def __init__(self, num_classes, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
        self.num_classes = num_classes

    def forward(self, logits, targets):
        log_probs = F.log_softmax(logits, dim=-1)
        one_hot = torch.zeros_like(log_probs).scatter(1, targets.unsqueeze(1), 1.0)
        smooth_targets = (1 - self.smoothing) * one_hot + self.smoothing / self.num_classes
        loss = -(smooth_targets * log_probs).sum(dim=-1)
        return loss.mean()

logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

ls_loss = LabelSmoothingLoss(10, smoothing=0.1)(logits, targets)
ce_loss = nn.CrossEntropyLoss()(logits, targets)

assert ls_loss.item() > 0
# With smoothing=0, should approximate CE
ls_loss_0 = LabelSmoothingLoss(10, smoothing=0.0)(logits, targets)
assert torch.allclose(ls_loss_0, ce_loss, atol=1e-5)
print(f'Label smoothing loss: {ls_loss.item():.4f}  CE loss: {ce_loss.item():.4f}')

### P12. Implement Gradient Accumulation

Simulate larger batch sizes without more GPU memory.

**How to approach:**
- Accumulate gradients over `k` mini-batches
- Call `optimizer.step()` every `k` steps
- Divide loss by `k` for correct scaling

In [ ]:
model = SimpleMLP(10, 32, 5)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

accumulation_steps = 4
dataset = SyntheticDataset(64, 10, 5)
loader = DataLoader(dataset, batch_size=4, shuffle=True)

model.train()
optimizer.zero_grad()
for i, (x, y) in enumerate(loader):
    loss = criterion(model(x), y) / accumulation_steps
    loss.backward()

    if (i + 1) % accumulation_steps == 0:
        optimizer.step()
        optimizer.zero_grad()

print(f'Gradient accumulation: effective batch size = {4 * accumulation_steps}')

### P13. Implement a Complete CNN (Tesla-style)

**How to approach:**
- Conv layers with batch norm, ReLU, pooling
- Flatten and use linear head
- Count parameters and verify shapes at each stage

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),   # -> (16, 32, 32)
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.MaxPool2d(2),                    # -> (16, 16, 16)
            nn.Conv2d(16, 32, 3, padding=1),   # -> (32, 16, 16)
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),                    # -> (32, 8, 8)
        )
        self.classifier = nn.Sequential(
            nn.Linear(32 * 8 * 8, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(1)
        return self.classifier(x)

cnn = SimpleCNN(num_classes=10)
x = torch.randn(4, 3, 32, 32)
out = cnn(x)

assert out.shape == (4, 10)
total = sum(p.numel() for p in cnn.parameters())
print(f'CNN output: {out.shape}  Total params: {total:,}')

### P14. Implement Cosine Similarity Loss (NVIDIA-style)

**How to approach:**
- Normalize embeddings to unit vectors
- Compute dot product as similarity
- Use margin-based or contrastive formulation

In [ ]:
def cosine_similarity_loss(z1, z2, temperature=0.5):
    """NT-Xent style contrastive loss (SimCLR)."""
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)
    N = z1.shape[0]

    z = torch.cat([z1, z2], dim=0)        # (2N, D)
    sim = (z @ z.T) / temperature          # (2N, 2N)

    # Mask out self-similarity
    mask = ~torch.eye(2 * N, dtype=torch.bool, device=z.device)
    sim = sim.masked_select(mask).reshape(2 * N, -1)

    # Positive pairs: (i, i+N) and (i+N, i)
    labels = torch.cat([torch.arange(N, 2*N), torch.arange(N)]).to(z.device)
    # Adjust labels for removed diagonal
    labels = labels - (labels > torch.arange(2*N, device=z.device)).long()

    return F.cross_entropy(sim, labels)

z1 = torch.randn(32, 128, requires_grad=True)
z2 = torch.randn(32, 128, requires_grad=True)

loss = cosine_similarity_loss(z1, z2)
assert loss.ndim == 0 and loss.item() > 0
loss.backward()
assert z1.grad is not None
print(f'Contrastive loss: {loss.item():.4f}')

### P15. Debugging a Broken Training Loop (Meta interview classic)

Spot and fix the bugs in this training loop.

**How to approach:**
- Check for missing `zero_grad`, wrong mode, missing `no_grad` during eval
- Verify loss decreases

In [ ]:
# BUGGY version (commented out) vs FIXED version

# Bug 1: missing zero_grad -> gradient accumulation
# Bug 2: model not in train mode
# Bug 3: calling loss.item() before backward (not a bug, but loss.backward() missing)
# Bug 4: not using no_grad for validation

model = SimpleMLP(10, 32, 5)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
train_data = SyntheticDataset(200, 10, 5)
val_data = SyntheticDataset(50, 10, 5)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=50)

# FIXED training loop
losses = []
for epoch in range(10):
    model.train()                          # Fix: set train mode
    epoch_loss = 0.0
    for x, y in train_loader:
        optimizer.zero_grad()              # Fix: zero gradients
        out = model(x)
        loss = F.cross_entropy(out, y)
        loss.backward()                    # Fix: compute gradients
        optimizer.step()
        epoch_loss += loss.item()
    losses.append(epoch_loss)

    # Validation
    model.eval()                           # Fix: eval mode
    with torch.no_grad():                  # Fix: no grad for val
        for x, y in val_loader:
            val_out = model(x)
            val_loss = F.cross_entropy(val_out, y)

assert losses[-1] < losses[0], 'Training loss should decrease'
print(f'Training loss: {losses[0]:.4f} -> {losses[-1]:.4f}')

---
## 11. Bonus: Quick-Fire Interview Snippets

In [ ]:
# --- Tensor shape manipulation patterns ---
# (batch, seq, heads, d_k) -> (batch, heads, seq, d_k)
x = torch.randn(2, 16, 8, 32)
y = x.permute(0, 2, 1, 3)
assert y.shape == (2, 8, 16, 32)

# --- Gather (used in embedding lookup, selecting from distributions) ---
probs = torch.randn(5, 10)
indices = torch.tensor([3, 7, 0, 2, 9]).unsqueeze(1)
gathered = torch.gather(probs, dim=1, index=indices)
assert gathered.shape == (5, 1)
for i in range(5):
    assert gathered[i, 0] == probs[i, indices[i, 0]]

# --- Repeat / expand ---
a = torch.tensor([1, 2, 3])
assert a.repeat(3).shape == (9,)
b = a.unsqueeze(0)  # (1, 3)
expanded = b.expand(4, 3)  # no copy
assert expanded.shape == (4, 3)

# --- torch.where ---
x = torch.tensor([-1.0, 2.0, -3.0, 4.0])
relu_manual = torch.where(x > 0, x, torch.zeros_like(x))
assert torch.allclose(relu_manual, F.relu(x))

# --- Einsum ---
A = torch.randn(3, 4)
B = torch.randn(4, 5)
assert torch.allclose(torch.einsum('ij,jk->ik', A, B), A @ B)

print('All quick-fire snippets verified.')

---
## Summary

| # | Topic | Key Takeaway |
|---|-------|--------------|
| 1 | Setup | Know your device, set seeds for reproducibility |
| 2 | Tensors | `shape`, `dtype`, `device`, NumPy interop shares memory |
| 3 | Indexing | `view` needs contiguous; `reshape` always works |
| 4 | Autograd | Dynamic graph, gradient accumulation, `no_grad`/`detach` |
| 5 | nn.Module | `parameters()`, `train()`/`eval()`, `Parameter` vs `buffer` |
| 6 | Loss/Optim | CE expects logits, 5-step train loop |
| 7 | Data | Custom `Dataset`/`DataLoader`, `num_workers`, `pin_memory` |
| 8 | Save/Load | Always use `state_dict`, save optimizer state too |
| 9 | FAQ | Computation graph, view vs reshape, eval vs no_grad, freeze layers, grad clipping |
| 10 | Practice | Custom autograd, Transformer, Focal Loss, Mixed Precision, Gradient Checkpointing |

**All cells passed `assert` checks — this notebook is machine-verifiable.**